# Project_Paru_CXR - Progress 3: Baseline EfficientNet-B0 (Kaggle)

Notebook ini bisa dijalankan **sekaligus** dengan **Save Version -> Save & Run All**.

**Sebelum menjalankan, atur di menu Settings / panel kanan:**
1. **Accelerator -> GPU T4 x2**
2. **Internet -> On** (wajib; perlu verifikasi nomor HP di akun Kaggle)
3. **Add Input -> kompetisi `rsna-pneumonia-detection-challenge`** (wajib)
4. *(Opsional)* Add Input -> dataset **Chest X-Ray Images (Pneumonia)** milik Paul Mooney untuk uji eksternal. Bila tidak ada, notebook mencoba mengunduhnya otomatis; bila gagal, uji eksternal dilewati.

Urutan: cek lingkungan -> ambil kode -> cari data -> konversi DICOM -> cek split -> data eksternal -> smoke test -> **latih baseline S1 (3 seed)** -> rekap -> simpan hasil.
Perkiraan waktu: ±1,5-2 jam di GPU T4.

In [ ]:
# 1. Cek lingkungan: GPU dan Internet (berhenti dengan pesan jelas bila belum diatur)
import os, socket, subprocess, sys, glob, shutil, time

def internet_ok(host="github.com"):
    try:
        socket.create_connection((host, 443), timeout=5).close()
        return True
    except OSError:
        return False

import torch
gpu = torch.cuda.is_available()
print("PyTorch :", torch.__version__)
print("GPU     :", torch.cuda.get_device_name(0) if gpu else "TIDAK ADA")
print("Internet:", "OK" if internet_ok() else "MATI")
if not internet_ok():
    raise SystemExit("Internet MATI. Nyalakan: Settings -> Internet -> On (verifikasi nomor HP bila diminta), lalu jalankan ulang.")
if not gpu:
    raise SystemExit("GPU belum aktif. Pilih Settings -> Accelerator -> GPU T4 x2, lalu jalankan ulang.")

In [ ]:
# 2. Ambil kode terbaru dari GitHub + pustaka yang dibutuhkan
REPO_URL = "https://github.com/monicabortang/Project_Paru_CXR.git"
REPO = "/kaggle/working/Project_Paru_CXR"

def run(cmd, check=True):
    """Jalankan perintah shell dan tampilkan output secara langsung."""
    print("\n$ " + cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Perintah gagal (kode {p.returncode}): {cmd}")
    return p.returncode

if os.path.isdir(os.path.join(REPO, ".git")):
    run(f"git -C {REPO} pull --ff-only")
else:
    run(f"git clone --depth 1 {REPO_URL} {REPO}")
os.chdir(REPO)
print("Folder kerja:", os.getcwd())

missing = []
for mod, pkg in [("timm", "timm"), ("pydicom", "pydicom"), ("cv2", "opencv-python-headless")]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    run("pip install -q " + " ".join(missing))
import timm, pydicom
print("timm", timm.__version__, "| pydicom", pydicom.__version__)

In [ ]:
# 3. Cari lokasi data otomatis (nama folder input Kaggle kadang berbeda)
def find_dir_with(filename, root="/kaggle/input"):
    hits = glob.glob(f"{root}/**/{filename}", recursive=True)
    return os.path.dirname(hits[0]) if hits else None

RSNA = find_dir_with("stage_2_train_labels.csv")
if RSNA is None:
    raise SystemExit("Data RSNA tidak ditemukan. Tambahkan: Add Input -> Competitions -> "
                     "rsna-pneumonia-detection-challenge (klik 'Join Competition' dulu).")
print("RSNA        :", RSNA)

def find_kermany():
    for c in glob.glob("/kaggle/input/**/chest_xray", recursive=True):
        if os.path.isdir(os.path.join(c, "test", "NORMAL")):
            return c
    try:  # coba unduh otomatis
        import kagglehub
        p = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
        for c in [os.path.join(p, "chest_xray"), p] + glob.glob(f"{p}/**/chest_xray", recursive=True):
            if os.path.isdir(os.path.join(c, "test", "NORMAL")):
                return c
    except Exception as e:
        print("Unduh Kermany otomatis gagal:", type(e).__name__, e)
    return None

KERMANY_SRC = find_kermany()
print("Kermany     :", KERMANY_SRC or "tidak tersedia -> uji eksternal dilewati")

DATA = "/kaggle/working/data"
SPLITS = f"{REPO}/splits"
EXT = "/kaggle/working/external/kermany"
print("Split repo  :", SPLITS, "(ada)" if os.path.exists(f"{SPLITS}/summary.csv") else "(BELUM ADA)")

In [ ]:
# 4. Konversi DICOM -> PNG 512 px + metadata (±15-25 menit; dilewati bila sudah ada)
if not os.path.exists(f"{DATA}/metadata.csv"):
    run(f"python -m src.prepare_data --rsna_dir '{RSNA}' --out_dir {DATA} --size 512 --workers 4")
else:
    print("metadata.csv sudah ada, dilewati.")
n_png = len(glob.glob(f"{DATA}/images/*.png"))
print("Jumlah PNG:", n_png)
assert n_png >= 26000, "Jumlah citra hasil konversi kurang dari 26.684"

In [ ]:
# 5. Pakai split dari laptop (sama dengan laporan Progress 2) dan pastikan cocok dengan data
import pandas as pd
if not os.path.exists(f"{SPLITS}/summary.csv"):
    print("Split repo tidak ada -> dibuat ulang dengan seed 42")
    run(f"python -m src.make_splits --meta {DATA}/metadata.csv --out {SPLITS} --seed 42")
meta_ids = set(pd.read_csv(f"{DATA}/metadata.csv").patientId)
for f in ["S1_all/train.csv", "S1_all/val.csv", "test/all.csv"]:
    ids = set(pd.read_csv(f"{SPLITS}/{f}").patientId)
    miss = len(ids - meta_ids)
    print(f"{f:20s} n={len(ids):6d}  tidak ditemukan={miss}")
    assert miss == 0, f"Ada citra di {f} yang tidak ada di data hasil konversi"
print(pd.read_csv(f"{SPLITS}/summary.csv").head(8).to_string(index=False))

In [ ]:
# 6. Data uji eksternal Kermany (opsional)
EXTARG = ""
if KERMANY_SRC:
    if not os.path.exists(f"{EXT}/kermany.csv"):
        run(f"python -m src.external kermany --src '{KERMANY_SRC}' --out {EXT} --subsets test")
    EXTARG = f"--external kermany={EXT}/kermany.csv"
print("Argumen eksternal:", EXTARG or "(tidak ada)")

In [ ]:
# 7. Smoke test GPU (±2-3 menit): memastikan pelatihan jalan sebelum run panjang
run(f"python scripts/run_experiments.py --group smoke --models efficientnet_b0 --splits {SPLITS} --img_root {DATA}")

In [ ]:
# 8. PELATIHAN BASELINE: EfficientNet-B0, skenario S1, seed 0/1/2 (±1-1,5 jam)
t0 = time.time()
run(f"python scripts/run_experiments.py --group baseline --splits {SPLITS} --img_root {DATA} {EXTARG}")
print(f"Selesai dalam {(time.time() - t0) / 60:.1f} menit")

In [ ]:
# 9. Rekap hasil + tampilkan kurva
run("python -m src.summarize --results results/all_results.csv --runs runs --out results/tables")
from IPython.display import Image, display
r = pd.read_csv("results/all_results.csv")
cols = ["model", "seed", "test", "n", "prevalence", "auc", "auc_lo", "auc_hi", "macro_f1", "sensitivity", "specificity"]
display(r[r.run_tag == "S1_all"][cols].round(4))
for s in (0, 1, 2):
    p = f"runs/S1_all/efficientnet_b0/seed{s}/curves.png"
    if os.path.exists(p):
        display(Image(p))

In [ ]:
# 10. Simpan hasil untuk diunduh (tab Output setelah run selesai)
run("zip -qr /kaggle/working/hasil_progress3.zip results runs -x '*.pt'")
best = "runs/S1_all/efficientnet_b0/seed0/best.pt"
if os.path.exists(best):
    shutil.copy(best, "/kaggle/working/best_effnet_s1_seed0.pt")
# hapus PNG hasil konversi dari output agar unduhan ringan (bisa dibuat ulang kapan saja)
shutil.rmtree(f"{DATA}/images", ignore_errors=True)
print(os.listdir("/kaggle/working"))
print("SELESAI. Unduh hasil_progress3.zip dan best_effnet_s1_seed0.pt dari tab Output.")